# CSV Chunking

## 1. Direct Vector Embedding Approach
- Process: Convert CSV rows/columns → Text chunks → Vector embeddings → Vector database
- Best for: Smaller datasets with clear semantic meaning
- Tools: LangChain's CSVLoader, LlamaIndex's CSVNodeParser
- Advantage: Simplest implementation with minimal preprocessing
## 2. CSV to JSON Conversion
- Process: CSV → JSON → Text chunks → Vector embeddings → Vector database
- Best for: Complex CSV structures with nested relationships
- Tools: Pandas for conversion, any RAG framework for processing
- Advantage: Better preservation of data relationships and structure
## 3. Structured Query Approach
- Process: CSV → Pandas DataFrame → LLM-generated SQL-like queries → Results to LLM
- Best for: Analytical questions requiring precise data manipulation
- Tools: Pandas, LangChain agents
- Advantage: Maintains data integrity, excellent for numerical analysis
## 4. Hybrid Chunking Strategy
- Process: CSV → Multiple chunking strategies → Combined vector database
- Best for: Complex datasets with varied query patterns
- Tools: LlamaIndex, Haystack
- Advantage: Balances between semantic search and structured data integrity
## 5. Summarization-First Approach
- Process: CSV → LLM summarization → Vector embeddings of summaries → Original data retrieval
- Best for: Very large CSV files with clear thematic sections
- Tools: Any LLM for summarization, vector database for storage
- Advantage: Reduces storage requirements while maintaining context

## 1. Direct Vector Embedding Approach

In [1]:
import pandas as pd

df = pd.read_csv("customer.csv")

def row_to_text(row):
    return (
        f"Customer ID {row['customer_id']} "
        f"is {row['name']} from {row['city']}. "
        f"Age is {row['age']} years. "
        f"Purchase amount is ${row['purchase_amount']}."
    )

documents = df.apply(row_to_text, axis=1).tolist()



In [2]:
documents

['Customer ID 101 is John from New York. Age is 28 years. Purchase amount is $500.',
 'Customer ID 102 is Alice from Chicago. Age is 35 years. Purchase amount is $1200.',
 'Customer ID 103 is Bob from Boston. Age is 42 years. Purchase amount is $800.']

In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-m3", device="cpu")

embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11832.50it/s]


In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="customers",
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE
    )
)

True

In [6]:
points = []

for idx, (text, vector) in enumerate(zip(documents, embeddings)):

    points.append(
        PointStruct(
            id=idx,
            vector=vector.tolist(),
            payload={
                "text": text,
                "customer_id": int(df.iloc[idx]["customer_id"]),
                "name": str(df.iloc[idx]["name"]),
                "city": str(df.iloc[idx]["city"]),
                "age": int(df.iloc[idx]["age"]),
                "purchase_amount": float(
                    df.iloc[idx]["purchase_amount"]
                )
            }
        )
    )

client.upsert(
    collection_name="customers",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [8]:
query = "customers with high purchases"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

In [10]:
results = client.query_points(
    collection_name="customers",
    query=query_embedding.tolist(),
    limit=3
)

print(results)

points=[ScoredPoint(id=2, version=0, score=0.6109242454756989, payload={'text': 'Customer ID 103 is Bob from Boston. Age is 42 years. Purchase amount is $800.', 'customer_id': 103, 'name': 'Bob', 'city': 'Boston', 'age': 42, 'purchase_amount': 800.0}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=0, score=0.5949853675040405, payload={'text': 'Customer ID 102 is Alice from Chicago. Age is 35 years. Purchase amount is $1200.', 'customer_id': 102, 'name': 'Alice', 'city': 'Chicago', 'age': 35, 'purchase_amount': 1200.0}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=0, version=0, score=0.5748373312834694, payload={'text': 'Customer ID 101 is John from New York. Age is 28 years. Purchase amount is $500.', 'customer_id': 101, 'name': 'John', 'city': 'New York', 'age': 28, 'purchase_amount': 500.0}, vector=None, shard_key=None, order_value=None)]


In [11]:
for hit in results.points:
    print(hit.payload)

{'text': 'Customer ID 103 is Bob from Boston. Age is 42 years. Purchase amount is $800.', 'customer_id': 103, 'name': 'Bob', 'city': 'Boston', 'age': 42, 'purchase_amount': 800.0}
{'text': 'Customer ID 102 is Alice from Chicago. Age is 35 years. Purchase amount is $1200.', 'customer_id': 102, 'name': 'Alice', 'city': 'Chicago', 'age': 35, 'purchase_amount': 1200.0}
{'text': 'Customer ID 101 is John from New York. Age is 28 years. Purchase amount is $500.', 'customer_id': 101, 'name': 'John', 'city': 'New York', 'age': 28, 'purchase_amount': 500.0}


## 3. Structured Query Approach

In [13]:
import pandas as pd

df = pd.read_csv("customer.csv")

In [27]:
prompt = """You are a pandas query generator.

Available columns:

customer_id : integer
name : string
city : string
age : integer
purchase_amount : float

Return only a valid pandas expression nothing else no colons nothing at all.

# Output Schema:
df[df['']] 

Question:
Which customers spent more than 1000?"""

In [28]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"]

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

response = client.chat.completions.create(
    model="meta/llama-3.3-70b-instruct",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(
    response.choices[0].message.content
)

df[df['purchase_amount'] > 1000]


In [29]:
query = response.choices[0].message.content

In [ ]:
# query = "df[df['purchase_amount'] > 1000]"

In [30]:
allowed_globals = {
    "df": df
}

result = eval(
    query,
    {"__builtins__": {}},
    allowed_globals
)

In [20]:
result

,customer_id,name,city,age,purchase_amount
1,102,Alice,Chicago,35,1200


# CSV Chunking Strategies for RAG Systems

## The Core Problem

Unlike PDFs and text documents, CSV files contain **structured data**.

Example:

| Product | Region | Revenue |
| ------- | ------ | ------- |
| iPhone  | Europe | 200M    |
| Mac     | Europe | 40M     |
| iPad    | Asia   | 30M     |

A naive chunking strategy may split rows arbitrarily:

```text
Chunk 1
--------
| Product | Region | Revenue |
| iPhone | Europe | 200M |

Chunk 2
--------
| Mac | Europe | 40M |
| iPad | Asia | 30M |
```

Problems:

* Header information is lost.
* Relationships between rows disappear.
* Retrieval quality decreases.
* Embeddings struggle to understand raw tabular structure.

---

# Why Embedding Entire CSV Files Is Usually Wrong

Suppose:

```text
1,000,000 rows
```

Embedding the entire CSV:

```text
CSV
 ↓
Single Embedding
```

is impossible because:

* Token limits are exceeded.
* Embeddings become noisy.
* Retrieval precision becomes poor.

Additionally, users often ask:

```text
What was the revenue of Mac in Europe?
What is the average sales value?
Top 10 products by revenue?
```

These are database-style questions rather than semantic search questions.

---

# Strategy 1: Row-Based Chunking

Store each row independently.

Example:

```json
{
  "product": "iPhone",
  "region": "Europe",
  "revenue": "200M"
}
```

Converted to text:

```text
Product: iPhone
Region: Europe
Revenue: 200M
```

Advantages:

* Easy retrieval.
* Good for lookup-style questions.

Disadvantages:

* Loses global context.
* Poor for aggregation questions.

---

# Strategy 2: Group-Based Chunking

Group rows by a meaningful dimension.

Example:

```text
Europe Sales
```

| Product | Revenue |
| ------- | ------- |
| iPhone  | 200M    |
| Mac     | 40M     |

Chunk:

```text
Sales data for Europe.

Product Revenue
iPhone 200M
Mac 40M
```

Advantages:

* Better context.
* Fewer chunks.

Disadvantages:

* Requires partitioning logic.

---

# Strategy 3: Table Summary Chunking (Modern RAG)

Instead of embedding raw tables, generate a natural language summary.

Original Table:

| Product | Revenue |
| ------- | ------- |
| iPhone  | 200B    |
| Mac     | 40B     |
| iPad    | 30B     |

Summary:

```text
Apple revenue table.

Products:
- iPhone
- Mac
- iPad

Highest revenue product is iPhone.
Revenue values range from 30B to 200B.
```

Embed:

```text
Summary
```

Store:

```json
{
  "summary": "...",
  "table_id": "tbl_001"
}
```

Advantages:

* Embeddings work much better on natural language.
* Retrieval quality improves significantly.

---

# Fetch Original Table Pattern

After retrieval finds the summary:

```text
User Query
     ↓
Vector Search
     ↓
Summary Found
     ↓
table_id
     ↓
Fetch Original Table
     ↓
Send To LLM
```

Example:

```json
{
  "table_id": "tbl_001"
}
```

Lookup:

```markdown
| Product | Revenue |
|----------|----------|
| iPhone | 200B |
| Mac | 40B |
| iPad | 30B |
```

This step is called:

```text
Fetch Original Table
```

---

# What If The Table Has 1,000,000 Rows?

Do NOT fetch the entire table.

Bad:

```text
Vector Search
 ↓
Retrieve Summary
 ↓
Send 1M Rows To LLM
```

Impossible due to context limits.

---

# Industry Solution: Treat Large CSVs As Databases

For large datasets:

```text
CSV
 ↓
Database
```

Examples:

* PostgreSQL
* DuckDB
* SQLite
* BigQuery

User asks:

```text
What was Mac revenue in Europe in Q2 2025?
```

Generated SQL:

```sql
SELECT revenue
FROM sales
WHERE product='Mac'
AND region='Europe'
AND quarter='Q2'
AND year=2025;
```

Database returns:

```text
42,000,000
```

Only the result is sent to the LLM.

---

# Hierarchical Table Retrieval

Large tables are often indexed at multiple levels.

```text
Sales Table
    ↓
Region Summaries
    ↓
Country Summaries
    ↓
Monthly Summaries
    ↓
Rows
```

Example:

```text
Revenue Table
      ↓
Europe
      ↓
Germany
      ↓
Q2 2025
      ↓
Matching Rows
```

This progressively narrows retrieval.

---

# Parent-Child Chunking For Tables

Same concept as text RAG.

Parent:

```text
Full Table
```

Child:

```text
Table Summary
```

Index:

```text
Summary
```

Retrieve:

```text
Summary
 ↓
Full Table
 ↓
LLM
```

For very large tables:

Parent:

```text
Regional Partition
```

Child:

```text
Regional Summary
```

instead of using the entire table.

---

# Recommended Production Architecture

## Small CSV (< 1,000 Rows)

```text
CSV
 ↓
Markdown Table
 ↓
Summary Generation
 ↓
Embedding
 ↓
Vector DB
```

Retrieve:

```text
Summary
 ↓
Original Table
 ↓
LLM
```

---

## Medium CSV (1,000 - 100,000 Rows)

```text
CSV
 ↓
Partition By Category
 ↓
Summary Per Partition
 ↓
Embedding
 ↓
Vector DB
```

Retrieve:

```text
Summary
 ↓
Partition
 ↓
LLM
```

---

## Large CSV (100,000+ Rows)

```text
CSV
 ↓
Database
 ↓
Text-to-SQL
 ↓
Results
 ↓
LLM
```

Avoid vector retrieval as the primary access mechanism.

---

# Final Rule

Use:

```text
Vector Search
```

for:

* Unstructured text
* Reports
* Documentation
* PDFs

Use:

```text
Text-to-SQL
```

for:

* Large CSVs
* Analytics
* Aggregations
* Filtering
* Sorting
* Joins

The larger and more structured the table becomes, the more valuable database querying becomes compared to vector search.
